# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))

# 建立sequence to sequence 模型

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 256
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.LSTMCell(self.hidden)
        self.decoder_cell = tf.keras.layers.LSTMCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        '''
        完成sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好
        '''
        enc_emb = self.embed_layer(enc_ids)
        _, enc_h, enc_c = self.encoder(enc_emb)

        dec_emb = self.embed_layer(dec_ids)
        dec_out, _, _ = self.decoder(dec_emb, initial_state=[enc_h, enc_c])
        logits = self.dense(dec_out)
        return logits
    
    
#     @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        _, enc_h, enc_c = self.encoder(enc_emb)
        
        return [enc_h, enc_c]
    
    def get_next_logits(self, x, state):
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.decoder_cell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        return logits, state

    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        logits, state = self.get_next_logits(x, state)
        out = tf.argmax(logits, axis=-1)
        return out, state

# Loss函数以及训练逻辑

In [4]:
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    curriculum = [max(2, seqlen - 6), max(2, seqlen - 4), max(2, seqlen - 2), seqlen]
    schedule = [500, 500, 500, 2000]
    step = 0
    for cur_len, rounds in zip(curriculum, schedule):
        for _ in range(rounds):
            _, enc_x, dec_x, y = get_batch(64, cur_len)
            loss = train_one_step(model, optimizer, enc_x, dec_x, y)
            step += 1
            if step % 500 == 0:
                print('step', step, 'len', cur_len, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.001)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=10)

step 500 len 4 : loss 0.024967704
step 1000 len 6 : loss 0.022012377
step 1500 len 8 : loss 0.036587816
step 2000 len 10 : loss 0.05865792
step 2500 len 10 : loss 0.03931279
step 3000 len 10 : loss 0.047340125
step 3500 len 10 : loss 0.026546631


<tf.Tensor: shape=(), dtype=float32, numpy=0.026546631008386612>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def clone_state(state):
        return [tf.identity(item) for item in state]

    def decode_one(init_state, steps=10, beam_width=15):
        beams = [(0.0, tf.zeros(shape=[1], dtype=tf.int32), clone_state(init_state), [])]
        for _ in range(steps):
            candidates = []
            for score, cur_token, state, seq in beams:
                logits, next_state = model.get_next_logits(cur_token, state)
                log_probs = tf.nn.log_softmax(logits, axis=-1)[0].numpy()
                top_tokens = np.argsort(log_probs)[-beam_width:][::-1]
                for token in top_tokens:
                    candidates.append((
                        score + float(log_probs[token]),
                        tf.constant([int(token)], dtype=tf.int32),
                        clone_state(next_state),
                        seq + [int(token)],
                    ))
            candidates.sort(key=lambda item: item[0], reverse=True)
            beams = candidates[:beam_width]

        best_seq = beams[0][3]
        return ''.join([chr(idx + ord('A') - 1) for idx in best_seq])
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    preds = []
    for i in range(len(batched_examples)):
        state = model.encode(enc_x[i:i+1])
        preds.append(decode_one(state, steps=enc_x.shape[1]))
    return preds, batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('OJRJVPYRJP', 'PJRYPVJRJO'), ('TMBQEUMGEB', 'BEGMUEQBMT'), ('TXWDHXIXOJ', 'JOXIXHDWXT'), ('AQFRBZJXJT', 'TJXJZBRFQA'), ('FOGFXWONSM', 'MSNOWXFGOF'), ('JFDXWBGIZO', 'OZIGBWXDFJ'), ('LANGISRXAH', 'HAXRSIGNAL'), ('PNANQVXZPP', 'PPZXVQNANP'), ('RPAXJGRADV', 'VDARGJXAPR'), ('DQQJWZJDHE', 'EHDJZWJQQD'), ('TYVLHJRYCN', 'NCYRJHLVYT'), ('BPPDRECEQK', 'KQECERDPPB'), ('SPGQJPYTRW', 'WRTYPJQGPS'), ('JCSUKCMCOA', 'AOCMCKUSCJ'), ('CHJHSTOUHT', 'THUOTSHJHC'), ('UUJKIIVVOL', 'LOVVIIKJUU'), ('EFSHNPWJJN', 'NJJWPNHSFE'), ('FBYLMKDHFL', 'LFHDKMLYBF'), ('TAWUZCKVMS', 'SMVKCZUWAT'), ('RHJUSOLWGM', 'MGWLOSUJHR'), ('KJBFSOLFEI', 'IEFLOSFBJK'), ('DAIZCSIEYP', 'PYEISCZIAD'), ('RDKFQBWUTW', 'WTUWBQFKDR'), ('FSERJCSDWH', 'HWDSCJRESF'), ('RULYUEBKSM', 'MSKBEUYLUR'), ('XQFRGCUSBE', 'EBSUCGRFQX'), ('YJSTXNTXPI', 'IPXTNXTSJY

In [ ]:
# 原版没有 attention 的版本只有约18%正确率，远不如 attention 版本，于是我改用了 LSTM 并且调整了部分训练策略，现在测试正确率显著提高了，但是也说明了attention的重要性